# Exploratory Data Analysis: KKBox churn features

This notebook profiles the 59 model features of the KKBox churn task and their relationship with `is_churn`.
Most features are engineered from three raw sources (members, transactions and daily listening logs);
the analysis runs on the final exported table, where every customer (`msno`) appears in exactly one row,
so each statistic below is computed per customer rather than per raw record.

The features end up in three groups: nominal categorical codes (`city`, `registered_via`,
`last_payment_method_id`), binary flags, and numeric features (discrete counts or continuous values).
Each group uses a measure suited to its type. All results are descriptive; observed associations are not
interpreted as causal.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

# February development cohort (cutoff 2017-01-31). Use "test" for the March cohort.
COHORT = "train_feb"

ON_KAGGLE = Path("/kaggle/input").exists()
DATA_CANDIDATES = [
    Path("/kaggle/input/dataset-final"),
    Path("export-tabular-v5"),
    Path("../../w1/export-tabular-v5"),
    Path("../../../export-tabular-v5"),
]
DATA_DIR = next(
    (path.resolve() for path in DATA_CANDIDATES if (path / f"{COHORT}.parquet").exists()),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError(
        f"{COHORT}.parquet not found in: {', '.join(map(str, DATA_CANDIDATES))}. "
        "Generate it with workshops/w1/w1_data_preprocessing.ipynb."
    )

# The 59 model features and the nominal ones come from the w1 preprocessing state;
# on Kaggle the export metadata provides the same catalogue.
STATE_PATH = Path("../../w1/artifacts/sherlock_w1/preprocessing/development.json")
state_path = STATE_PATH if STATE_PATH.exists() else DATA_DIR / "metadata.json"
state = json.loads(state_path.read_text())
FEATURES = state["features"]
CATEGORICAL = state["categorical"]

GRAPHICS_DIR = (Path("/kaggle/working") if ON_KAGGLE else Path(".")) / "figures" / "eda"
GRAPHICS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(DATA_DIR / f"{COHORT}.parquet", columns=["msno", "is_churn"] + FEATURES)
assert df["msno"].is_unique, "expected one row per customer"
y = df["is_churn"].astype(int)

print(f"Source: {DATA_DIR / f'{COHORT}.parquet'}")
print(f"Feature catalogue: {state_path}")
print(f"Shape: {df.shape[0]:,} customers x {len(FEATURES)} features (+ msno, is_churn)")
display(df.head())

## Feature catalogue

Each feature is assigned a source domain (member, transactions, listening logs) and a type:

| Type | Rule | Relationship measure |
|---|---|---|
| categorical | Listed as nominal in the preprocessing state | Cramér's V, churn rate per level |
| binary | Two distinct values | Churn rate per value |
| discrete | Integer values with at most 31 distinct levels | Churn rate per value |
| continuous | Everything else | Box plots by class, churn rate per decile |
| constant | One distinct value | Excluded from correlations |

Missing values are kept: the export leaves some transaction statistics null for customers without
transactions, and these rows are dropped pairwise in each measure.

In [ ]:
DOMAIN_STARTS = {"member": "city", "transactions": "is_cancel_change_flag", "logs": "sum_last_month_unq"}
starts = sorted((FEATURES.index(first), name) for name, first in DOMAIN_STARTS.items())
DOMAIN = {}
for (start, name), (end, _) in zip(starts, starts[1:] + [(len(FEATURES), None)]):
    DOMAIN.update({feature: name for feature in FEATURES[start:end]})


def feature_type(series, name):
    values = series.dropna()
    n_unique = values.nunique()
    if name in CATEGORICAL:
        return "categorical"
    if n_unique <= 1:
        return "constant"
    if n_unique == 2:
        return "binary"
    if n_unique <= 31 and np.allclose(values, np.round(values)):
        return "discrete"
    return "continuous"


catalogue = pd.DataFrame({
    "domain": [DOMAIN[f] for f in FEATURES],
    "type": [feature_type(df[f], f) for f in FEATURES],
    "dtype": [str(df[f].dtype) for f in FEATURES],
    "n_unique": [df[f].nunique() for f in FEATURES],
    "missing_pct": [df[f].isna().mean() * 100 for f in FEATURES],
}, index=pd.Index(FEATURES, name="feature"))

TYPES = {kind: catalogue.index[catalogue["type"].eq(kind)].tolist()
         for kind in ["categorical", "binary", "discrete", "continuous", "constant"]}
NUMERIC = TYPES["binary"] + TYPES["discrete"] + TYPES["continuous"]

display(pd.crosstab(catalogue["domain"], catalogue["type"], margins=True))
display(catalogue.round(2))
if TYPES["constant"]:
    print("Constant features (no information about churn):", TYPES["constant"])

## Target distribution

`is_churn = 1` denotes customers who did not renew within the label month. The class counts, percentages and
majority-to-minority ratio quantify the imbalance. Because churners are a small minority, later plots show the
churn rate within each group against the overall rate (dashed line) instead of raw counts.

In [ ]:
import matplotlib.pyplot as plt

churn_labels = ["No", "Yes"]
churn_colors = ["#4b8f8c", "#d66b4e"]

target_counts = y.map({0: "No", 1: "Yes"}).value_counts().reindex(churn_labels)
target_percentages = target_counts / len(df) * 100
imbalance_ratio = target_counts.max() / target_counts.min()
BASE_RATE = y.mean()

display(pd.DataFrame({"customers": target_counts, "percent": target_percentages.round(2)}))
print(f"Majority-to-minority ratio: {imbalance_ratio:.2f}:1 | overall churn rate: {BASE_RATE:.2%}")

ax = target_counts.plot(kind="bar", color=churn_colors, rot=0)
ax.set(title="Customer churn target distribution", xlabel="is_churn", ylabel="Customers")
ax.bar_label(ax.containers[0], fmt="{:,.0f}")
plt.tight_layout()
plt.savefig(GRAPHICS_DIR / "target_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

## Numerical feature statistics

Count, mean, standard deviation and quantiles for every numeric feature. Many activity and payment aggregates
are heavily right-skewed, so the median and the 99th percentile are more informative than the mean and maximum.

In [ ]:
summary = df[NUMERIC].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T
summary.insert(0, "type", catalogue.loc[NUMERIC, "type"])
summary.insert(0, "domain", catalogue.loc[NUMERIC, "domain"])
display(summary.round(3))

## Correlation of numeric features with churn

For binary, discrete and continuous features:

- **Pearson** with the 0/1 target is the point-biserial correlation: the linear difference in means between classes.
- **Spearman** uses ranks, so it is robust to the skewed distributions and captures any monotonic relationship.
- **ROC AUC** is the probability that a random churner has a higher value than a random non-churner
  (0.5 = no separation; values below 0.5 mean lower values go with churn).

Positive correlations mean higher values are associated with churn. Features are sorted by absolute Spearman correlation.

In [ ]:
from sklearn.metrics import roc_auc_score


def univariate_auc(values):
    mask = values.notna()
    return roc_auc_score(y[mask], values[mask])


numeric_corr = pd.DataFrame({
    "domain": catalogue.loc[NUMERIC, "domain"],
    "type": catalogue.loc[NUMERIC, "type"],
    "pearson": df[NUMERIC].corrwith(y),
    "spearman": df[NUMERIC].corrwith(y, method="spearman"),
    "roc_auc": [univariate_auc(df[f]) for f in NUMERIC],
})
numeric_corr = numeric_corr.reindex(numeric_corr["spearman"].abs().sort_values(ascending=False).index)
display(numeric_corr.round(4))

order = numeric_corr.index[::-1]
positions = np.arange(len(order))
bar_height = 0.4
fig, ax = plt.subplots(figsize=(9, 0.25 * len(order) + 1.5))
ax.barh(positions + bar_height / 2, numeric_corr.loc[order, "spearman"], bar_height,
        label="Spearman", color="#2f5d8a")
ax.barh(positions - bar_height / 2, numeric_corr.loc[order, "pearson"], bar_height,
        label="Pearson", color="#a9b8c6")
ax.set_yticks(positions, labels=order, fontsize=8)
ax.axvline(0, color="black", linewidth=0.8)
ax.set(title=f"Correlation with is_churn ({COHORT})", xlabel="Correlation coefficient")
ax.grid(axis="x", alpha=0.25)
ax.set_axisbelow(True)
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(GRAPHICS_DIR / "churn_numeric_correlations.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

## Categorical features and churn

The nominal codes have no order, so a correlation coefficient does not apply. **Cramér's V** (bias-corrected,
0 = independent, 1 = fully determined) measures the strength of association from the contingency table.
The plots show the churn rate of the most frequent levels; rarer levels are pooled into `other` and missing
values appear as `missing`.

In [ ]:
from scipy.stats import chi2_contingency

MAX_LEVELS = 12


def as_levels(series):
    labels = series.astype("Int64").astype("string").fillna("missing")
    top = labels.value_counts().index[:MAX_LEVELS]
    return labels.where(labels.isin(top), "other")


def cramers_v(table):
    chi2 = chi2_contingency(table, correction=False)[0]
    n = table.to_numpy().sum()
    rows, cols = table.shape
    phi2 = max(0.0, chi2 / n - (rows - 1) * (cols - 1) / (n - 1))
    rows_c = rows - (rows - 1) ** 2 / (n - 1)
    cols_c = cols - (cols - 1) ** 2 / (n - 1)
    return np.sqrt(phi2 / min(rows_c - 1, cols_c - 1))


categorical_assoc = pd.DataFrame({
    "domain": catalogue.loc[TYPES["categorical"], "domain"],
    "n_levels": catalogue.loc[TYPES["categorical"], "n_unique"],
    "cramers_v": [
        cramers_v(pd.crosstab(df[f].astype("Int64").astype("string").fillna("missing"), y))
        for f in TYPES["categorical"]
    ],
}).sort_values("cramers_v", ascending=False)
display(categorical_assoc.round(4))

fig, axes = plt.subplots(len(TYPES["categorical"]), 1,
                         figsize=(11, 3.6 * len(TYPES["categorical"])), squeeze=False)
for axis, feature in zip(axes.flat, TYPES["categorical"]):
    levels = as_levels(df[feature])
    stats = y.groupby(levels).agg(["mean", "size"]).sort_values("size", ascending=False)
    bars = axis.bar(stats.index, stats["mean"] * 100, color=churn_colors[1])
    axis.axhline(BASE_RATE * 100, color="black", linestyle="--", linewidth=1, label="overall rate")
    axis.bar_label(bars, labels=[f"{r:.1%}\nn={n:,}" for r, n in zip(stats["mean"], stats["size"])],
                   padding=1, fontsize=7)
    axis.set_title(f"{feature} (levels sorted by customers)")
    axis.set_ylabel("Churn rate (%)")
    axis.set_ylim(0, stats["mean"].max() * 100 * 1.3)
    axis.grid(axis="y", alpha=0.25)
    axis.set_axisbelow(True)
    axis.legend(fontsize=8)
fig.tight_layout()
fig.savefig(GRAPHICS_DIR / "churn_categorical_relationships.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

## Binary and discrete features and churn

For flags and low-cardinality counts, the churn rate per value shows the full relationship directly.
Bars are annotated with the churn rate and the number of customers; the dashed line is the overall churn rate.
Features are grouped four per image.

In [ ]:
def plot_rate_by_value(features, filename, title):
    for index in range(0, len(features), 4):
        group = features[index:index + 4]
        fig, axes = plt.subplots(1, len(group), figsize=(4.5 * len(group), 4), squeeze=False)
        for axis, feature in zip(axes.flat, group):
            values = df[feature].astype("Float64").round().astype("Int64").astype("string").fillna("missing")
            stats = y.groupby(values).agg(["mean", "size"])
            numeric_keys = pd.to_numeric(stats.index.to_series(), errors="coerce").fillna(np.inf)
            stats = stats.iloc[np.argsort(numeric_keys.to_numpy(), kind="stable")]
            bars = axis.bar(stats.index, stats["mean"] * 100, color=churn_colors[1])
            axis.axhline(BASE_RATE * 100, color="black", linestyle="--", linewidth=1)
            if len(stats) <= 4:
                axis.bar_label(bars, labels=[f"{r:.1%}\nn={n:,}" for r, n in zip(stats["mean"], stats["size"])],
                               padding=1, fontsize=7)
            axis.set_ylim(0, stats["mean"].max() * 100 * 1.3)
            axis.tick_params(axis="x", labelsize=7, rotation=90 if len(stats) > 12 else 0)
            axis.set_title(feature)
            axis.set_ylabel("Churn rate (%)")
            axis.grid(axis="y", alpha=0.25)
            axis.set_axisbelow(True)
        fig.suptitle(title)
        fig.tight_layout()
        output_path = GRAPHICS_DIR / f"{filename}_{index // 4 + 1:02d}.png"
        fig.savefig(output_path, dpi=300, bbox_inches="tight")
        print(f"Saved: {output_path}")
        plt.show()
        plt.close(fig)


plot_rate_by_value(TYPES["binary"], "churn_binary_relationships", "Churn rate by flag value")
plot_rate_by_value(TYPES["discrete"], "churn_discrete_relationships", "Churn rate by value")

## Continuous features and churn

Each column shows one feature: the top panel compares the distribution by class (box plots without outliers,
as in the Telco EDA), and the bottom panel shows the churn rate per decile of the feature, which exposes
non-linear relationships that a single correlation coefficient can hide. Deciles collapse when many customers
share a value (for example zero activity), so some features have fewer than ten bins. Features are grouped
four per image in catalogue order.

In [ ]:
continuous = TYPES["continuous"]
for index in range(0, len(continuous), 4):
    group = continuous[index:index + 4]
    fig, axes = plt.subplots(2, len(group), figsize=(4.5 * len(group), 7), squeeze=False)
    for column, feature in enumerate(group):
        values = df[feature]
        box_axis, rate_axis = axes[0, column], axes[1, column]

        class_values = [values[y.eq(label)].dropna().to_numpy() for label in (0, 1)]
        boxplots = box_axis.boxplot(class_values, patch_artist=True, showfliers=False)
        for box, color in zip(boxplots["boxes"], churn_colors):
            box.set_facecolor(color)
            box.set_alpha(0.8)
        box_axis.set_xticks([1, 2], labels=churn_labels)
        box_axis.set_title(feature)
        box_axis.grid(axis="y", alpha=0.25)

        mask = values.notna()
        deciles = pd.qcut(values[mask], 10, duplicates="drop")
        stats = pd.DataFrame({"x": values[mask], "y": y[mask], "bin": deciles}).groupby(
            "bin", observed=True).agg(median=("x", "median"), rate=("y", "mean"))
        rate_axis.plot(np.arange(len(stats)), stats["rate"] * 100, marker="o", color=churn_colors[1])
        rate_axis.axhline(BASE_RATE * 100, color="black", linestyle="--", linewidth=1)
        rate_axis.set_xticks(np.arange(len(stats)), labels=[f"{m:.3g}" for m in stats["median"]],
                             rotation=60, fontsize=7)
        rate_axis.set_xlabel("Decile median")
        rate_axis.set_ylabel("Churn rate (%)")
        rate_axis.set_ylim(bottom=0)
        rate_axis.grid(alpha=0.25)
    fig.tight_layout()
    output_path = GRAPHICS_DIR / f"churn_continuous_relationships_{index // 4 + 1:02d}.png"
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    print(f"Saved: {output_path}")
    plt.show()
    plt.close(fig)

## Correlation between features

Many engineered features are built from the same raw fields (sums, averages and last values of the same
transaction columns, or consecutive months of the same listening counts), so some are near-duplicates.
The Spearman matrix below, computed on a stratified sample for speed, shows these blocks; pairs with
|ρ| ≥ 0.9 are listed as redundancy candidates.

In [ ]:
SAMPLE_SIZE = 200_000
SEED = 42

sample = df.groupby("is_churn", group_keys=False).sample(
    frac=min(1.0, SAMPLE_SIZE / len(df)), random_state=SEED
)
feature_corr = sample[NUMERIC].corr(method="spearman")

fig, ax = plt.subplots(figsize=(14, 12))
image = ax.imshow(feature_corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(np.arange(len(NUMERIC)), labels=NUMERIC, rotation=90, fontsize=6)
ax.set_yticks(np.arange(len(NUMERIC)), labels=NUMERIC, fontsize=6)
ax.set_title(f"Spearman correlation between numeric features ({len(sample):,} customers)")
fig.colorbar(image, ax=ax, fraction=0.04, pad=0.02, label="Spearman ρ")
fig.tight_layout()
fig.savefig(GRAPHICS_DIR / "feature_correlation_matrix.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

upper = feature_corr.where(np.triu(np.ones(feature_corr.shape, dtype=bool), k=1))
redundant = upper.stack().rename("spearman").reset_index()
redundant.columns = ["feature_a", "feature_b", "spearman"]
redundant = redundant[redundant["spearman"].abs().ge(0.9)]
redundant = redundant.reindex(redundant["spearman"].abs().sort_values(ascending=False).index)
redundant["churn_spearman_a"] = numeric_corr.loc[redundant["feature_a"], "spearman"].to_numpy()
redundant["churn_spearman_b"] = numeric_corr.loc[redundant["feature_b"], "spearman"].to_numpy()
print(f"Pairs with |rho| >= 0.9: {len(redundant)}")
display(redundant.round(3).reset_index(drop=True))

## Unified ranking: mutual information

Correlation coefficients and Cramér's V are not on the same scale, so the final ranking uses **mutual
information** with the target, which applies to every feature type and also captures non-monotonic
relationships. It is estimated on the same stratified sample; categorical, binary and discrete features are
treated as discrete, and missing values are filled with the median (numeric) or a separate code (categorical).
The table joins all the measures computed above.

In [ ]:
from sklearn.feature_selection import mutual_info_classif

ranked = TYPES["categorical"] + NUMERIC
X_sample = sample[ranked].copy()
for feature in TYPES["categorical"]:
    X_sample[feature] = X_sample[feature].fillna(-1)
X_sample[NUMERIC] = X_sample[NUMERIC].fillna(X_sample[NUMERIC].median())
discrete_mask = [feature not in TYPES["continuous"] for feature in ranked]

mutual_info = pd.Series(
    mutual_info_classif(X_sample.to_numpy(dtype=float), sample["is_churn"].to_numpy(),
                        discrete_features=discrete_mask, random_state=SEED),
    index=ranked, name="mutual_info",
)

feature_ranking = (
    catalogue.loc[ranked, ["domain", "type", "missing_pct"]]
    .join(numeric_corr[["pearson", "spearman", "roc_auc"]])
    .join(categorical_assoc[["cramers_v"]])
    .join(mutual_info)
    .sort_values("mutual_info", ascending=False)
)
feature_ranking.insert(0, "rank", np.arange(1, len(feature_ranking) + 1))
display(feature_ranking.round(4))
feature_ranking.to_csv(GRAPHICS_DIR / "feature_churn_ranking.csv")

type_colors = {"categorical": "#7a5195", "binary": "#ef5675", "discrete": "#ffa600", "continuous": "#2f5d8a"}
order = feature_ranking.index[::-1]
fig, ax = plt.subplots(figsize=(9, 0.25 * len(order) + 1.5))
ax.barh(order, feature_ranking.loc[order, "mutual_info"],
        color=[type_colors[t] for t in feature_ranking.loc[order, "type"]])
ax.tick_params(axis="y", labelsize=8)
ax.set(title=f"Mutual information with is_churn ({COHORT})", xlabel="Mutual information (nats)")
ax.grid(axis="x", alpha=0.25)
ax.set_axisbelow(True)
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in type_colors.values()],
          labels=list(type_colors), loc="lower right")
fig.tight_layout()
fig.savefig(GRAPHICS_DIR / "churn_mutual_information.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

for domain, group in feature_ranking.groupby("domain", sort=False):
    print(f"Top {domain} features:", ", ".join(group.index[:5]))

## Findings

Complete after running the notebook, using `feature_ranking` and the figures in `figures/eda/`:

- **Target:** overall churn rate and imbalance ratio of the cohort.
- **Strongest signals:** the top features by mutual information and their direction (sign of Spearman,
  ROC AUC above or below 0.5), and which domain (member, transactions, logs) they come from.
- **Non-linear patterns:** continuous features whose decile curve is not monotonic, where the
  correlation coefficient understates the relationship.
- **Categorical levels:** levels of `city`, `registered_via` and `last_payment_method_id` with churn rates far
  from the overall rate, and their size.
- **Redundancy:** feature pairs with |ρ| ≥ 0.9 that carry the same information.
- **Weak or constant features:** features with near-zero mutual information and correlation.

These are associations in this cohort, not causal effects.